# 수요 필터 통과 행정동 QR 공백 시각화

`마스터_A_동.csv` 또는 `03_qr_gap_filtered.csv`를 업로드하면 됩니다. 전체 43개 동 파일은 방문객 또는 외국인 카드 결제 건수의 하위 25%를 제외하는 OR 방식으로 수요 필터를 다시 계산합니다.

In [ ]:
# 셀 1. CSV 업로드
from google.colab import files

uploaded = files.upload()
FILE_NAME = next(iter(uploaded))
print('업로드 파일:', FILE_NAME)

In [ ]:
# 셀 2. 데이터 불러오기 및 열 설정
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

for encoding in ['utf-8-sig', 'cp949']:
    try:
        df = pd.read_csv(FILE_NAME, encoding=encoding)
        break
    except UnicodeDecodeError:
        continue

REGION = '행정동명'
VISIT = '방문'
PAYMENT = '건수_음식소매'
QR_COUNT = 'QR가맹점_음식소매'
BUSINESS_COUNT = '사업체수_음식소매'
QR_RATE = 'QR가맹률_음식소매'
QR_GAP = 'QR공백률_음식소매'

required = [REGION, QR_COUNT, BUSINESS_COUNT]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f'필요한 열이 없습니다: {missing}')

print(f'원본 행 수: {len(df)}개')
display(df.head())

In [ ]:
# 셀 3. 수요 필터 적용: 방문 또는 결제건수 하위 25% 제외(OR)
Q_CUT = 0.25

if '수요필터_통과' not in df.columns:
    if VISIT in df.columns and PAYMENT in df.columns and len(df) > 37:
        visit_cut = df[VISIT].quantile(Q_CUT)
        payment_cut = df[PAYMENT].quantile(Q_CUT)
        df['방문_통과'] = (df[VISIT] > visit_cut).astype(int)
        df['결제_통과'] = (df[PAYMENT] > payment_cut).astype(int)
        df['수요필터_통과'] = (df['방문_통과'].eq(1) | df['결제_통과'].eq(1)).astype(int)
        print(f'방문 기준값: {visit_cut:,.0f}')
        print(f'결제건수 기준값: {payment_cut:,.0f}')
    else:
        # 이미 37개 동으로 필터링된 결과 파일로 간주
        df['수요필터_통과'] = 1

filtered = df.loc[df['수요필터_통과'].eq(1)].copy()
excluded = df.loc[df['수요필터_통과'].eq(0), REGION].tolist()

print(f'통과: {len(filtered)}개 / 제외: {len(df) - len(filtered)}개')
if excluded:
    print('제외 지역:', excluded)

In [ ]:
# 셀 4. QR가맹률·공백률 계산과 임계값 비교
filtered[QR_RATE] = np.where(
    filtered[BUSINESS_COUNT] > 0,
    filtered[QR_COUNT] / filtered[BUSINESS_COUNT],
    np.nan,
)
filtered[QR_GAP] = 1 - filtered[QR_RATE]

if filtered[QR_RATE].isna().any():
    bad = filtered.loc[filtered[QR_RATE].isna(), REGION].tolist()
    raise ValueError(f'사업체 수가 0이거나 QR가맹률을 계산할 수 없는 지역: {bad}')

mean_rate = filtered[QR_RATE].mean()
median_rate = filtered[QR_RATE].median()

def classify(rate):
    if rate < mean_rate:
        return '두 기준 모두 부족'
    if rate < median_rate:
        return '기준 민감'
    return '중앙값 이상'

filtered['QR분류'] = filtered[QR_RATE].map(classify)
filtered['QR가맹률(%)'] = filtered[QR_RATE] * 100
filtered['QR공백률(%)'] = filtered[QR_GAP] * 100
filtered = filtered.sort_values(QR_RATE).reset_index(drop=True)

comparison = pd.DataFrame({
    '기준': ['중앙값(본 분석)', '평균(민감도 분석)'],
    'QR가맹률 임계값': [median_rate, mean_rate],
    'QR 부족 지역 수': [
        int((filtered[QR_RATE] < median_rate).sum()),
        int((filtered[QR_RATE] < mean_rate).sum()),
    ],
})
comparison.style.format({'QR가맹률 임계값': '{:.2%}'})

In [ ]:
# 셀 5. 분류 결과 확인
order = ['두 기준 모두 부족', '기준 민감', '중앙값 이상']
counts = filtered['QR분류'].value_counts().reindex(order, fill_value=0)

print(f'분석 대상: {len(filtered)}개 동')
print(f'중앙값: {median_rate:.2%} / 평균: {mean_rate:.2%}')
print(f'두 기준의 차이: {(median_rate - mean_rate) * 100:.2f}%p')
print(counts.to_string())
print('\n기준에 따라 분류가 달라지는 동')
display(
    filtered.loc[
        filtered['QR분류'].eq('기준 민감'),
        [REGION, 'QR가맹률(%)', 'QR공백률(%)', QR_COUNT, BUSINESS_COUNT],
    ].style.format({'QR가맹률(%)': '{:.2f}', 'QR공백률(%)': '{:.2f}'})
)

In [ ]:
# 셀 6. 행정동별 QR가맹률 시각화
color_map = {
    '두 기준 모두 부족': '#D55E00',
    '기준 민감': '#E69F00',
    '중앙값 이상': '#0072B2',
}

hover_columns = {}
for col in [QR_GAP, QR_COUNT, BUSINESS_COUNT, VISIT, PAYMENT]:
    if col in filtered.columns:
        hover_columns[col] = ':.2%' if col == QR_GAP else ':,.0f'

fig = px.bar(
    filtered,
    x='QR가맹률(%)',
    y=REGION,
    orientation='h',
    color='QR분류',
    category_orders={
        'QR분류': order,
        REGION: filtered[REGION].tolist(),
    },
    color_discrete_map=color_map,
    text=filtered['QR가맹률(%)'].map(lambda x: f'{x:.1f}%'),
    hover_data=hover_columns,
    title='수요 필터 통과 행정동의 QR 가맹률',
    labels={'QR가맹률(%)': 'QR 가맹률 (%)', REGION: '', 'QR분류': ''},
)

fig.update_traces(textposition='outside', cliponaxis=False)
fig.add_vline(
    x=mean_rate * 100,
    line_dash='dash',
    line_color='#E69F00',
    annotation_text=f'평균 {mean_rate:.2%}',
    annotation_position='top left',
)
fig.add_vline(
    x=median_rate * 100,
    line_dash='solid',
    line_color='#D55E00',
    annotation_text=f'중앙값 {median_rate:.2%}',
    annotation_position='top right',
)
fig.update_layout(
    height=max(750, len(filtered) * 26 + 140),
    template='plotly_white',
    legend_title_text='',
    bargap=0.20,
    margin=dict(l=90, r=90, t=100, b=60),
    xaxis=dict(range=[max(0, filtered['QR가맹률(%)'].min() - 3), filtered['QR가맹률(%)'].max() + 2], ticksuffix='%'),
    yaxis=dict(autorange='reversed'),
)
fig.show()

In [ ]:
# 셀 7. 결과 저장 및 다운로드
RESULT_CSV = '03_qr_gap_filtered_or25.csv'
RESULT_HTML = 'qr_gap_37_dongs.html'

filtered.to_csv(RESULT_CSV, index=False, encoding='utf-8-sig')
fig.write_html(RESULT_HTML, include_plotlyjs='cdn')
print('저장 완료:', RESULT_CSV, RESULT_HTML)

# 필요할 때 아래 두 줄의 주석을 해제하면 파일을 내려받습니다.
# files.download(RESULT_CSV)
# files.download(RESULT_HTML)

In [ ]:
# 셀 8. 지역별 QR가맹률·QR공백률 비교 시각화
# 셀 1~4를 먼저 실행한 후 이 셀을 실행하세요.

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 앞 셀의 수요 필터 통과 데이터 복사
qr_compare = filtered.copy()

# 앞 셀을 수정했더라도 필요한 비율을 다시 확인·계산
if QR_RATE not in qr_compare.columns:
    qr_compare[QR_RATE] = np.where(
        qr_compare[BUSINESS_COUNT] > 0,
        qr_compare[QR_COUNT] / qr_compare[BUSINESS_COUNT],
        np.nan,
    )

if QR_GAP not in qr_compare.columns:
    qr_compare[QR_GAP] = 1 - qr_compare[QR_RATE]

if qr_compare[[QR_RATE, QR_GAP]].isna().any().any():
    problem_regions = qr_compare.loc[
        qr_compare[[QR_RATE, QR_GAP]].isna().any(axis=1),
        REGION,
    ].tolist()
    raise ValueError(f'QR 비율을 계산할 수 없는 지역: {problem_regions}')

# 수요 필터를 통과한 지역만을 기준으로 임계값 계산
mean_rate_compare = qr_compare[QR_RATE].mean()
median_rate_compare = qr_compare[QR_RATE].median()
mean_gap_compare = 1 - mean_rate_compare
median_gap_compare = 1 - median_rate_compare

qr_compare['QR가맹률(%)'] = qr_compare[QR_RATE] * 100
qr_compare['QR공백률(%)'] = qr_compare[QR_GAP] * 100

# QR공백률이 큰 지역부터 정렬
qr_compare = qr_compare.sort_values(
    'QR공백률(%)',
    ascending=False,
).reset_index(drop=True)

# 지역별 수치 표
display_columns = [
    REGION,
    'QR가맹률(%)',
    'QR공백률(%)',
    QR_COUNT,
    BUSINESS_COUNT,
]

if 'QR분류' in qr_compare.columns:
    display_columns.append('QR분류')

display(
    qr_compare[display_columns].style.format({
        'QR가맹률(%)': '{:.2f}',
        'QR공백률(%)': '{:.2f}',
        QR_COUNT: '{:,.0f}',
        BUSINESS_COUNT: '{:,.0f}',
    })
)

# QR가맹률과 QR공백률을 나란히 비교하는 두 개의 그래프
fig_compare = make_subplots(
    rows=1,
    cols=2,
    shared_yaxes=True,
    horizontal_spacing=0.06,
    subplot_titles=(
        '지역별 QR가맹률',
        '지역별 QR공백률',
    ),
)

custom_data = np.column_stack([
    qr_compare[QR_COUNT],
    qr_compare[BUSINESS_COUNT],
])

fig_compare.add_trace(
    go.Bar(
        x=qr_compare['QR가맹률(%)'],
        y=qr_compare[REGION],
        orientation='h',
        name='QR가맹률',
        marker_color='#0072B2',
        text=qr_compare['QR가맹률(%)'].map(lambda x: f'{x:.1f}%'),
        textposition='outside',
        cliponaxis=False,
        customdata=custom_data,
        hovertemplate=(
            '<b>%{y}</b><br>'
            'QR가맹률: %{x:.2f}%<br>'
            'QR가맹점: %{customdata[0]:,.0f}곳<br>'
            '음식·소매 사업체: %{customdata[1]:,.0f}곳'
            '<extra></extra>'
        ),
    ),
    row=1,
    col=1,
)

fig_compare.add_trace(
    go.Bar(
        x=qr_compare['QR공백률(%)'],
        y=qr_compare[REGION],
        orientation='h',
        name='QR공백률',
        marker_color='#D55E00',
        text=qr_compare['QR공백률(%)'].map(lambda x: f'{x:.1f}%'),
        textposition='outside',
        cliponaxis=False,
        customdata=custom_data,
        hovertemplate=(
            '<b>%{y}</b><br>'
            'QR공백률: %{x:.2f}%<br>'
            'QR가맹점: %{customdata[0]:,.0f}곳<br>'
            '음식·소매 사업체: %{customdata[1]:,.0f}곳'
            '<extra></extra>'
        ),
    ),
    row=1,
    col=2,
)

# 중앙값은 본 분석 기준, 평균은 민감도 분석 기준
fig_compare.add_vline(
    x=median_rate_compare * 100,
    line_color='#D55E00',
    line_width=2,
    annotation_text=f'중앙값 {median_rate_compare:.2%}',
    annotation_position='top left',
    row=1,
    col=1,
)
fig_compare.add_vline(
    x=mean_rate_compare * 100,
    line_color='#E69F00',
    line_dash='dash',
    line_width=2,
    annotation_text=f'평균 {mean_rate_compare:.2%}',
    annotation_position='top right',
    row=1,
    col=1,
)
fig_compare.add_vline(
    x=median_gap_compare * 100,
    line_color='#D55E00',
    line_width=2,
    annotation_text=f'중앙값 {median_gap_compare:.2%}',
    annotation_position='top left',
    row=1,
    col=2,
)
fig_compare.add_vline(
    x=mean_gap_compare * 100,
    line_color='#E69F00',
    line_dash='dash',
    line_width=2,
    annotation_text=f'평균 {mean_gap_compare:.2%}',
    annotation_position='top right',
    row=1,
    col=2,
)

fig_compare.update_xaxes(
    range=[0, 100],
    ticksuffix='%',
    title_text='QR가맹률 (%)',
    row=1,
    col=1,
)
fig_compare.update_xaxes(
    range=[0, 100],
    ticksuffix='%',
    title_text='QR공백률 (%)',
    row=1,
    col=2,
)

fig_compare.update_yaxes(
    autorange='reversed',
    title_text='',
    row=1,
    col=1,
)

fig_compare.update_layout(
    title={
        'text': '수요 필터 통과 지역의 QR가맹률과 QR공백률',
        'x': 0.5,
        'xanchor': 'center'
    },
    height=max(850, len(qr_compare) * 27 + 180),
    template='plotly_white',
    showlegend=True,
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.07,
        xanchor='left',
        x=0,
    ),
    margin=dict(l=90, r=70, t=150, b=60),
    bargap=0.20,
)

fig_compare.show()

# 선택 사항: 인터랙티브 그래프 저장
COMPARE_HTML = 'qr_rate_and_gap_by_dong.html'
fig_compare.write_html(COMPARE_HTML, include_plotlyjs='cdn')
print(f'저장 완료: {COMPARE_HTML}')